In [ ]:
import pandas as pd
import kagglehub as kaggle
from kagglehub import KaggleDatasetAdapter
import math # Perlu diimport untuk fungsi math.ceil

In [6]:
'''
- Mengunduh dan memuat dataset langsung dari Kaggle ke dalam format Pandas DataFrame.
- kagglehub secara otomatis mengelola proses download dan cache di direktori sistem, sehingga folder proyek Anda tetap bersih dari file data yang berukuran besar.
'''
df = kaggle.dataset_load(
          KaggleDatasetAdapter.PANDAS,
          "marketneutral/quandl-wiki-prices-us-equites",
          "WIKI_PRICES.csv"
)
# Menampilkan 5 baris pertama dari DataFrame untuk melakukan verifikasi 
# bahwa data telah berhasil dimuat dengan benar dan mengecek struktur kolomnya.
print(df.head())

  ticker        date   open   high    low  close      volume  ex-dividend  \
0      A  1999-11-18  45.50  50.00  40.00  44.00  44739900.0          0.0   
1      A  1999-11-19  42.94  43.00  39.81  40.38  10897100.0          0.0   
2      A  1999-11-22  41.31  44.00  40.06  44.00   4705200.0          0.0   
3      A  1999-11-23  42.50  43.63  40.25  40.25   4274400.0          0.0   
4      A  1999-11-24  40.13  41.94  40.00  41.06   3464400.0          0.0   

   split_ratio   adj_open   adj_high    adj_low  adj_close  adj_volume  
0          1.0  31.041951  34.112034  27.289627  30.018590  44739900.0  
1          1.0  29.295415  29.336350  27.160002  27.548879  10897100.0  
2          1.0  28.183363  30.018590  27.330562  30.018590   4705200.0  
3          1.0  28.995229  29.766161  27.460188  27.460188   4274400.0  
4          1.0  27.378319  28.613174  27.289627  28.012803   3464400.0  


In [7]:
df = df[['adj_open', 'adj_high','adj_low','adj_close', 'adj_volume',]]
'''
- df['HL_PCT'] <- data frame for calculate percentage high - percentage low on that day. so, this is almost like a percentage volatility.
''' 
df['HL_PCT'] = (df['adj_high'] - df['adj_close']) / df['adj_close'] * 100.0
df['PCT_change'] = (df['adj_close'] - df['adj_open']) / df['adj_open'] * 100.0

df = df[['adj_close', 'HL_PCT', 'PCT_change', 'adj_volume']]
print(df.head())

   adj_close     HL_PCT  PCT_change  adj_volume
0  30.018590  13.636364   -3.296703  44739900.0
1  27.548879   6.488361   -5.961807  10897100.0
2  30.018590   0.000000    6.511740   4705200.0
3  27.460188   8.397516   -5.294118   4274400.0
4  28.012803   2.143205    2.317468   3464400.0


In [ ]:
forecast_column = 'adj_close' # Menentukan kolom mana yang ingin diprediksi harganya

# Mengisi baris yang kosong (NaN) dengan angka yang sangat kecil (-99999)
# agar dikenali sebagai outlier oleh machine learning dan tidak perlu menghapus baris data
df.fillna(-99999, inplace=True)

,adj_close,HL_PCT,PCT_change,adj_volume,label
0,30.018590,13.636364,-3.296703,44739900.0,2.170821
1,27.548879,6.488361,-5.961807,10897100.0,2.187764
2,30.018590,0.000000,6.511740,4705200.0,2.139053
3,27.460188,8.397516,-5.294118,4274400.0,2.189882
4,28.012803,2.143205,2.317468,3464400.0,2.200471
...,...,...,...,...,...
13574754,10.917659,0.000000,0.832968,148600.0,9.640000
13574755,10.739653,2.209945,-1.630435,375200.0,9.665000
13574756,10.708799,0.576241,0.557165,428300.0,9.417500
13574757,10.827469,0.284963,0.817680,117900.0,9.362500


In [25]:
# Menentukan seberapa jauh ke depan (dalam jumlah baris) kita ingin memprediksi.
# Contoh: 0.01 berarti 1% dari total data.
forecast_out = int(math.ceil(0.01 * len(df)))

# Membuat kolom 'label' yang berisi data harga masa depan.
# .shift(-forecast_out) menggeser data harga ke atas (negatif) sehingga 
# baris hari ini akan memiliki label harga dari X hari ke depan.
df['label'] = df[forecast_column].shift(-forecast_out)

# Menghapus baris yang memiliki nilai NaN pada kolom label (karena data di akhir 
# dataframe tidak memiliki data masa depan yang cukup untuk diprediksi).
df.dropna(inplace=True)

# Menampilkan 5 baris terakhir untuk memeriksa apakah label sudah terisi dengan benar
df.tail()

,adj_close,HL_PCT,PCT_change,adj_volume,label
12276759,19.293164,0.400000,-0.088810,1247400.0,15.009539
12276760,19.524682,0.441501,0.935829,1353000.0,15.102455
12276761,19.248837,2.060009,-1.325674,1083600.0,15.102455
12276762,19.498822,1.635721,-0.571429,1131600.0,15.052423
12276763,19.524682,1.059603,0.443459,664900.0,15.545594
